# NumPy 2 — indexing, broadcasting, vectorisation

**What's in here**
- Basic, fancy and boolean indexing; `np.where`, `np.select`, `np.clip`
- Broadcasting rules with shapes printed explicitly; the `(n,)` vs `(n,1)` trap
- `axis` semantics for reductions
- Vectorising a Python loop and timing it
- `cumsum`, `cumprod`, `diff`, `searchsorted`, `argsort`, `argmax`, `unique`
- Float comparison: `isclose` / `allclose` instead of `==`

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)
print(np.__version__, pd.__version__)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df["consumption_mwh"].to_numpy()
temp = df["temp_c"].to_numpy()
price = df["price_eur_mwh"].to_numpy()
hour = df["time"].dt.hour.to_numpy()
print(df.shape)

1.26.4 2.3.3
(17520, 6)


## Basic indexing and slicing
`a[i]`, `a[i:j]`, `a[i:j:step]`, negative indices from the end. For 2-D use one bracket with a comma: `m[row, col]`, `m[:, col]`. `m[row][col]` works too but creates an intermediate array and can't be used with slices in the way you expect.

In [2]:
a = np.arange(10) * 10
print(a[0], a[-1], a[2:5], a[::3], a[::-1])

m = np.arange(12).reshape(3, 4)
print(m[1, 2], m[1], m[:, 2], m[0:2, 1:3], sep="\n")

0 90 [20 30 40] [ 0 30 60 90] [90 80 70 60 50 40 30 20 10  0]
6
[4 5 6 7]
[ 2  6 10]
[[1 2]
 [5 6]]


## Fancy indexing
Index with a list/array of integers to pick arbitrary positions (result is a copy). With 2-D arrays, `m[rows, cols]` pairs them elementwise — it does **not** give the sub-grid. Use `np.ix_` for the grid.

**Pitfall:** `m[[0, 2], [1, 3]]` returns 2 elements `(m[0,1], m[2,3])`, not a 2×2 block.

In [3]:
a = np.arange(10) * 10
print(a[[0, 3, 3, -1]])

m = np.arange(12).reshape(3, 4)
print("paired:", m[[0, 2], [1, 3]])
print("grid via ix_:\n", m[np.ix_([0, 2], [1, 3])])
print("rows only:\n", m[[0, 2]])

[ 0 30 30 90]
paired: [ 1 11]
grid via ix_:
 [[ 1  3]
 [ 9 11]]
rows only:
 [[ 0  1  2  3]
 [ 8  9 10 11]]


## Boolean indexing
A boolean array of the same shape selects elements. Combine conditions with `&`, `|`, `~` (not `and`/`or`) and wrap each condition in parentheses. `.sum()` on a mask counts True values, `.mean()` gives the fraction.

In [4]:
cold = temp < 0
print("cold hours:", cold.sum(), " fraction:", cold.mean().round(4))
print("mean consumption when cold vs not:", cons[cold].mean().round(0), cons[~cold].mean().round(0))

evening_cold = (hour >= 17) & (hour <= 20) & (temp < 2)
print("cold evenings:", evening_cold.sum(), cons[evening_cold].mean().round(0))

try:
    (temp < 0) and (hour > 17)
except ValueError as e:
    print("ValueError:", e)

cold hours: 1139  fraction: 0.065
mean consumption when cold vs not: 30058.0 29264.0
cold evenings: 153 37289.0
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


Assigning through a boolean mask is the idiomatic way to clean values (sentinels → NaN, cap outliers). It modifies the array in place.

In [5]:
p = price.copy()
print("negative prices:", (p < 0).sum())
p[p < 0] = 0.0                        # floor at zero, in place
print("negative prices after:", (p < 0).sum())

t = temp.copy()
t[t < -5] = np.nan                    # pretend anything below -5 is a sensor fault
print("NaNs introduced:", np.isnan(t).sum())

negative prices: 43
negative prices after: 0
NaNs introduced: 41


## `np.where`, `np.select`, `np.clip`
- `np.where(cond, a, b)` — vectorised if/else (with one argument it returns the indices where True)
- `np.select([c1, c2, ...], [v1, v2, ...], default)` — vectorised if/elif/else, first match wins
- `np.clip(x, lo, hi)` — cap values

In [6]:
peak = np.where((hour >= 16) & (hour <= 19), "peak", "offpeak")
print(np.unique(peak, return_counts=True))

print("indices of the 3 highest prices:", np.where(price > np.sort(price)[-3])[0])

season = np.select(
    [temp < 5, temp < 15, temp < 22],
    ["cold", "mild", "warm"],
    default="hot",
)
print(pd.Series(season).value_counts())

print("price clipped to [0, 200]:", np.clip(price, 0, 200)[:5], " max:", np.clip(price, 0, 200).max())

(array(['offpeak', 'peak'], dtype='<U7'), array([14600,  2920]))
indices of the 3 highest prices: [6237 6691]
mild    8058
cold    4806
warm    4253
hot      403
Name: count, dtype: int64
price clipped to [0, 200]: [81.83 88.21 84.71 70.92 60.78]  max: 200.0


## Broadcasting
Two arrays are compatible when, aligning shapes from the *right*, each pair of dimensions is equal or one of them is 1. The size-1 dimension is stretched. Scalars broadcast to anything.

```
(3, 4)  op  (4,)   -> (3, 4)     row vector applied to each row
(3, 4)  op  (3, 1) -> (3, 4)     column vector applied to each column
(3, 4)  op  (3,)   -> error!     3 != 4 on the last axis
```

In [7]:
m = np.arange(12).reshape(3, 4).astype(float)
row = np.array([10, 20, 30, 40])          # (4,)
col = np.array([[100], [200], [300]])     # (3, 1)

print((m + row).shape, "\n", m + row)
print((m + col).shape, "\n", m + col)
try:
    m + np.array([1, 2, 3])
except ValueError as e:
    print("ValueError:", e)

(3, 4) 
 [[10. 21. 32. 43.]
 [14. 25. 36. 47.]
 [18. 29. 40. 51.]]
(3, 4) 
 [[100. 101. 102. 103.]
 [204. 205. 206. 207.]
 [308. 309. 310. 311.]]
ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


Practical use: de-mean each column of a matrix (`X - X.mean(axis=0)`), or compute a full pairwise difference table with `a[:, None] - a[None, :]` — an outer operation without any loop.

In [8]:
X = np.column_stack([temp, cons, price])
Xc = X - X.mean(axis=0)                       # (n,3) - (3,) broadcasts along rows
print(Xc.mean(axis=0).round(8))               # ~0

Xz = (X - X.mean(axis=0)) / X.std(axis=0)     # z-score every column
print(Xz.std(axis=0))

a = np.array([1.0, 2.0, 4.0])
print(a[:, None] - a[None, :])                # pairwise differences, shape (3,3)

[-0.  0.  0.]
[1. 1. 1.]
[[ 0. -1. -3.]
 [ 1.  0. -2.]
 [ 3.  2.  0.]]


### The `(n,)` vs `(n,1)` trap
Subtracting a `(n,1)` column from an `(n,)` vector does **not** give `(n,)` — it broadcasts to an `(n, n)` matrix. This silently produces a huge (and wrong) array; with n = 17,520 that's 2.5 GB.

**Interview check:** "The residuals have shape (17520, 17520)" — where did that come from? → mixing `y` as a column vector (e.g. from `df[["y"]].values` or `reshape(-1,1)`) with predictions as a 1-D vector.

In [9]:
y = np.arange(5, dtype=float)                 # (5,)
y_col = y.reshape(-1, 1)                      # (5,1)
pred = y + 0.1                                # (5,)

print("y - pred     ->", (y - pred).shape)
print("y_col - pred ->", (y_col - pred).shape, "  <-- outer subtraction, wrong")
print("fix: ravel   ->", (y_col.ravel() - pred).shape)

y - pred     -> (5,)
y_col - pred -> (5, 5)   <-- outer subtraction, wrong
fix: ravel   -> (5,)


## `axis` semantics
`axis` is the dimension that gets **collapsed**. For a `(rows, cols)` matrix, `axis=0` collapses rows → one value per column; `axis=1` collapses columns → one value per row.

```
          axis=1 →
axis=0  [[ 0  1  2  3]      sum(axis=0) = [12 15 18 21]   (per column)
  ↓      [ 4  5  6  7]      sum(axis=1) = [ 6 22 38]      (per row)
         [ 8  9 10 11]]
```
`keepdims=True` keeps the collapsed axis as size 1 so the result broadcasts back against the original.

In [10]:
m = np.arange(12).reshape(3, 4)
print(m.sum(), m.sum(axis=0), m.sum(axis=1), sep="\n")
print(m.mean(axis=1, keepdims=True).shape)     # (3,1) -> can subtract from m directly

# hourly profile: reshape 2 years into (days, 24) and average over days (axis=0)
daily = cons.reshape(-1, 24)
print(daily.shape, "\nmean by hour of day:", daily.mean(axis=0).round(0)[:8], "...")

66
[12 15 18 21]
[ 6 22 38]
(3, 1)
(730, 24) 
mean by hour of day: [25430. 24372. 23875. 23610. 23883. 24989. 27230. 30060.] ...


## Vectorising a loop
Python loops over 17k elements are slow; numpy pushes the loop into C. Rule of thumb: if you're writing `for i in range(len(x))`, look for the array expression instead. Here: a heating-degree feature `max(15 - temp, 0)`.

In [11]:
def heating_loop(t):
    out = np.empty(len(t))
    for i in range(len(t)):
        out[i] = max(15 - t[i], 0)
    return out

def heating_vec(t):
    return np.maximum(15 - t, 0)

print(np.allclose(heating_loop(temp), heating_vec(temp)))

True


In [12]:
%timeit heating_loop(temp)
%timeit heating_vec(temp)

4.72 ms ± 46.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


16.1 µs ± 345 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Some loops genuinely can't be vectorised because each step depends on the previous result (recursions like an AR(1) or an EWMA). Then loop in Python or use pandas' `ewm`. Know the difference: elementwise or reduction → vectorise; sequential dependence → loop / specialised function.

In [13]:
# EWMA: y_t = a*x_t + (1-a)*y_{t-1}  -- sequential, loop is fine, or use pandas
alpha = 0.1
y = np.empty(len(cons)); y[0] = cons[0]
for i in range(1, len(cons)):
    y[i] = alpha * cons[i] + (1 - alpha) * y[i - 1]
print(np.allclose(y, pd.Series(cons).ewm(alpha=alpha, adjust=False).mean().to_numpy()))

True


## Cumulative operations and differences
`cumsum`/`cumprod` are running totals; `np.diff` gives `x[1:] - x[:-1]` (one element shorter!). Prepend to keep the length. Quant use: cumulative P&L, compounding returns, hourly changes.

In [14]:
x = np.array([1.0, 2.0, 3.0, 4.0])
print(np.cumsum(x), np.cumprod(x))
print(np.diff(x), np.diff(x).shape, "  vs prepend:", np.diff(x, prepend=np.nan))

rets = np.array([0.01, -0.02, 0.03])
print("compounded growth:", np.cumprod(1 + rets))

d_cons = np.diff(cons)
print("largest hourly jump in consumption:", d_cons.max().round(0), "at hour index", d_cons.argmax() + 1)

[ 1.  3.  6. 10.] [ 1.  2.  6. 24.]
[1. 1. 1.] (3,)   vs prepend: [nan  1.  1.  1.]
compounded growth: [1.01   0.9898 1.0195]
largest hourly jump in consumption: 4607.0 at hour index 15271


## `searchsorted` — binning and as-of lookups
Given a *sorted* array, `searchsorted(a, v)` returns the insertion position. It's the numpy way to bin values, or to find "the last observation at or before time t" (as-of join). `side="right"` matters for whether ties go left or right.

**Pitfall:** the input must be sorted; numpy won't check.

In [15]:
edges = np.array([0, 5, 10, 15, 20, 25])
print(np.searchsorted(edges, [3, 5, 12, 30]))
print(np.searchsorted(edges, [3, 5, 12, 30], side="right"))

# bin temperatures into bands and average consumption per band
band = np.searchsorted(edges, temp, side="right")
for b in np.unique(band):
    print(f"band {b}: temp<{edges[b] if b < len(edges) else 'inf'}  mean cons {cons[band == b].mean():,.0f}")

[1 1 3 6]
[1 2 3 6]
band 0: temp<0  mean cons 30,058
band 1: temp<5  mean cons 30,960
band 2: temp<10  mean cons 30,369
band 3: temp<15  mean cons 27,429
band 4: temp<20  mean cons 28,216
band 5: temp<25  mean cons 29,781
band 6: temp<inf  mean cons 30,142


## `argsort`, `argmax`, `argmin`
The `arg*` functions return *positions*, which you then use to index other arrays — e.g. "which hours had the 5 highest prices, and what was consumption then". `argmax` returns the *first* maximum only.

In [16]:
order = np.argsort(price)                 # ascending positions
top5 = order[-5:][::-1]
print("top-5 price positions:", top5)
print(df.loc[top5, ["time", "price_eur_mwh", "consumption_mwh"]])

print("argmax:", price.argmax(), price[price.argmax()])
print("sorted descending directly:", np.sort(price)[::-1][:5])

top-5 price positions: [ 6691  6237  7834  6448 12906]
                           time  price_eur_mwh  consumption_mwh
6691  2022-10-06 19:00:00+00:00         419.60          34087.5
6237  2022-09-17 21:00:00+00:00         379.06          28326.2
7834  2022-11-23 10:00:00+00:00         375.29          33647.3
6448  2022-09-26 16:00:00+00:00         361.10          31867.9
12906 2023-06-22 18:00:00+00:00         347.36          34185.6
argmax: 6691 419.6
sorted descending directly: [419.6  379.06 375.29 361.1  347.36]


## `np.unique`
Sorted distinct values; `return_counts=True` gives a frequency table, `return_inverse=True` gives integer codes (a poor man's label encoder).

In [17]:
dow = df["time"].dt.day_name().to_numpy()
vals, counts = np.unique(dow, return_counts=True)
print(dict(zip(vals, counts)))

codes_vals, codes = np.unique(dow, return_inverse=True)
print(codes[:10], "->", codes_vals[codes[:10]])

{'Friday': 2496, 'Monday': 2496, 'Saturday': 2520, 'Sunday': 2520, 'Thursday': 2496, 'Tuesday': 2496, 'Wednesday': 2496}
[2 2 2 2 2 2 2 2 2 2] -> ['Saturday' 'Saturday' 'Saturday' 'Saturday' 'Saturday' 'Saturday'
 'Saturday' 'Saturday' 'Saturday' 'Saturday']


## Comparing floats: `isclose` / `allclose`
Never use `==` on floats after arithmetic. `np.isclose(a, b, rtol=1e-5, atol=1e-8)` is elementwise, `np.allclose` is the all-reduce. Also `np.array_equal` for exact equality including shape. `np.isclose` treats NaN as unequal unless `equal_nan=True`.

**Interview check:** "Your reconciliation test fails on 3 rows out of a million" → floating-point noise; compare with a tolerance, and check for NaN.

In [18]:
a = 0.1 + 0.2
print(a == 0.3, np.isclose(a, 0.3))

x = cons / 3 * 3
print("exact equal:", np.array_equal(x, cons), "  allclose:", np.allclose(x, cons))
print("how many differ exactly:", (x != cons).sum())

print(np.isclose([np.nan], [np.nan]), np.isclose([np.nan], [np.nan], equal_nan=True))

False True
exact equal: False   allclose: True
how many differ exactly: 2954
[False] [ True]


## Quick reference

| Task | Code |
|---|---|
| if/else vectorised | `np.where(cond, a, b)` |
| if/elif/else | `np.select([c1, c2], [v1, v2], default)` |
| cap values | `np.clip(x, lo, hi)` |
| per-column stat | `X.mean(axis=0)` |
| per-row stat | `X.mean(axis=1)` |
| de-mean columns | `X - X.mean(axis=0)` |
| column vector | `x[:, None]` |
| pairwise / outer | `a[:, None] - a[None, :]` |
| bin into edges | `np.searchsorted(edges, x, side="right")` |
| top-k positions | `np.argsort(x)[-k:][::-1]` |
| frequency table | `np.unique(x, return_counts=True)` |
| float equality | `np.allclose(a, b)` |